In [6]:
import pandas as pd
import numpy as np

df=pd.read_csv('/content/orders.csv')

df['delivery_date']=pd.to_datetime(df['delivery_date'])

df['delay_days']=(pd.Timestamp.today()-df['delivery_date']).dt.days

df['is_delayed']=np.where(df['delay_days']>5,1,0)

print(df)

   order_id  supplier_id delivery_date     status  delay_days  is_delayed
0       101            1    2026-05-01  delivered          12           1
1       102            2    2026-05-12    delayed           1           0
2       103            1    2026-05-08  delivered           5           0
3       104            2    2026-05-03    delayed          10           1


In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       4 non-null      int64         
 1   supplier_id    4 non-null      int64         
 2   delivery_date  4 non-null      datetime64[ns]
 3   status         4 non-null      object        
 4   delay_days     4 non-null      int64         
 5   is_delayed     4 non-null      int64         
dtypes: datetime64[ns](1), int64(4), object(1)
memory usage: 324.0+ bytes
None


In [8]:
print(df.describe())

         order_id  supplier_id        delivery_date  delay_days  is_delayed
count    4.000000      4.00000                    4    4.000000     4.00000
mean   102.500000      1.50000  2026-05-06 00:00:00    7.000000     0.50000
min    101.000000      1.00000  2026-05-01 00:00:00    1.000000     0.00000
25%    101.750000      1.00000  2026-05-02 12:00:00    4.000000     0.00000
50%    102.500000      1.50000  2026-05-05 12:00:00    7.500000     0.50000
75%    103.250000      2.00000  2026-05-09 00:00:00   10.500000     1.00000
max    104.000000      2.00000  2026-05-12 00:00:00   12.000000     1.00000
std      1.290994      0.57735                  NaN    4.966555     0.57735


## week - 3

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark=SparkSession.builder.appName("supplychain").getOrCreate()

In [10]:
df=spark.read.csv(
'orders.csv',
header=True,
inferSchema=True
)

df.show()

+--------+-----------+-------------+---------+
|order_id|supplier_id|delivery_date|   status|
+--------+-----------+-------------+---------+
|     101|          1|   2026-05-01|delivered|
|     102|          2|   2026-05-12|  delayed|
|     103|          1|   2026-05-08|delivered|
|     104|          2|   2026-05-03|  delayed|
+--------+-----------+-------------+---------+



In [11]:
delayed=df.filter(col("status")=="delayed")

delayed.show()

+--------+-----------+-------------+-------+
|order_id|supplier_id|delivery_date| status|
+--------+-----------+-------------+-------+
|     102|          2|   2026-05-12|delayed|
|     104|          2|   2026-05-03|delayed|
+--------+-----------+-------------+-------+



In [12]:
result=delayed.groupBy("supplier_id").count()

result.show()

+-----------+-----+
|supplier_id|count|
+-----------+-----+
|          2|    2|
+-----------+-----+



In [13]:
result.write.csv(
'output',
header=True
)